# Find Candidate Induction Heads

Using the head embedding distances from notebook 03/04, find heads in other models
(pythia-1b, gemma-2b, etc.) that are nearby known GPT-2 small induction heads.

These candidates are then tested via ablation in notebook 06.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import numpy as np

# attention-motifs
from attention_motifs.attnpedia.attnpedia import AttentionPedia
from attention_motifs.features.analysis import DistanceTensorResult
from attention_motifs.ablation.candidates import (
    get_known_induction_heads,
    find_candidate_induction_heads,
    get_control_heads,
    save_candidates_to_file,
)

In [2]:
# config
pl.Config.set_tbl_rows(20)
PATH_BASE: Path = Path("../data/")

# Load distances and known heads

In [3]:
# load head distances
HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.read(
    PATH_BASE / "features" / "head_dists.zanj",
)
print(f"Loaded distances for {HEAD_DISTS.n_heads} heads")
print(f"Models: {set(h.split(':')[0] for h in HEAD_DISTS.cls_values)}")

FileNotFoundError: file not found: ../data/features/head_dists.zanj

In [ ]:
# load known induction heads
ATTNPEDIA = AttentionPedia()
KNOWN_INDUCTION = get_known_induction_heads(ATTNPEDIA)
print(f"Known induction heads ({len(KNOWN_INDUCTION)}):")
for head in KNOWN_INDUCTION:
    print(f"  {head}")

# Find candidate induction heads in other models

In [ ]:
# find candidates based on embedding proximity
CANDIDATES = find_candidate_induction_heads(
    HEAD_DISTS,
    reference_heads=KNOWN_INDUCTION,
    k_neighbors=20,
    exclude_reference_model=True,
    score_method="frequency",
)

print(f"Reference heads used: {len(CANDIDATES.reference_heads)}")
print(f"Models with candidates: {list(CANDIDATES.candidates_by_model.keys())}")

In [ ]:
# show top candidates per model
for model, candidates in CANDIDATES.candidates_by_model.items():
    print(f"\n{model}:")
    for head, score in candidates[:10]:
        print(f"  {head}: {score:.3f}")

In [ ]:
# convert to DataFrame for analysis
CANDIDATES_DF = CANDIDATES.to_dataframe()
CANDIDATES_DF

# Visualize candidate distribution

In [ ]:
# plot score distribution by model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of scores
for model in CANDIDATES.candidates_by_model.keys():
    model_df = CANDIDATES_DF.filter(pl.col("model") == model)
    axes[0].hist(model_df["score"].to_numpy(), alpha=0.5, label=model, bins=20)

axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Candidate Score Distribution by Model")
axes[0].legend()

# layer distribution of top candidates
TOP_N = 10
for model in CANDIDATES.candidates_by_model.keys():
    top_df = CANDIDATES_DF.filter(pl.col("model") == model).head(TOP_N)
    layers = top_df["layer"].to_numpy()
    axes[1].hist(layers, alpha=0.5, label=model, bins=range(0, 20))

axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Layer Distribution of Top {TOP_N} Candidates")
axes[1].legend()

plt.tight_layout()
plt.savefig("figures/ablation_candidates.pdf", bbox_inches="tight")

# Analyze nearest neighbors for each reference head

In [ ]:
# show which candidates appear for multiple reference heads
from collections import Counter

candidate_counts = Counter()
for ref_head, neighbors in CANDIDATES.all_neighbors.items():
    for neighbor, dist in neighbors:
        candidate_counts[neighbor] += 1

print("Heads appearing in top-K for multiple reference heads:")
for head, count in candidate_counts.most_common(20):
    if count > 1:
        print(f"  {head}: appears {count} times")

# Get control heads for baseline comparison

In [ ]:
# get control heads (far from induction heads) for each model
CONTROL_HEADS = {}
for model in CANDIDATES.candidates_by_model.keys():
    controls = get_control_heads(
        HEAD_DISTS, CANDIDATES, model, n_controls=5, method="far"
    )
    CONTROL_HEADS[model] = controls
    print(f"\n{model} control heads (far from induction):")
    for head in controls:
        print(f"  {head}")

# Save candidates for ablation study

In [ ]:
# save candidates to file
output_path = PATH_BASE / "ablation" / "candidates.json"
save_candidates_to_file(CANDIDATES, output_path)
print(f"Saved candidates to {output_path}")

# also save as CSV for easy viewing
CANDIDATES_DF.write_csv(PATH_BASE / "ablation" / "candidates.csv")